# CS781 D222: efficient legal decoder → legal NER

A new from-scratch efficient decoder: **222,079,872 parameters**. This notebook is the control and explanation layer for the included `legal_d` source package. Extract the complete source ZIP and open this notebook beside `configs`, `legal_d`, and `vendor`.

**No scores or training results are pre-filled.** Parameter count is model capacity; training-token count is data/compute spent. Strong scores require real held-out evaluation. Phase 1 closed October 6; its new ZIP is a benchmark artifact. Phase 2 deadline is October 11, 23:59 India time. Set a new deadline for later experiments.

Architecture mechanisms follow [Mistral 7B](https://arxiv.org/abs/2310.06825). More parameters without enough tokens can be undertrained: [Hoffmann et al.](https://arxiv.org/abs/2203.15556). Actual GPU hardware and kernels are logged. T4 portability does not mean a T4 run occurred.


In [ ]:
from pathlib import Path
import os,sys,json
ROOT=Path.cwd()
assert (ROOT/'legal_d/model.py').is_file(), 'Extract the source ZIP; run notebook from its architecture_d directory'
# stud1 has two GPUs; a single-GPU T4 notebook must set GPU_INDEX=0.
GPU_INDEX=1
os.environ['CUDA_VISIBLE_DEVICES']=str(GPU_INDEX)
os.environ['OMP_NUM_THREADS']='4'
for item in (ROOT,ROOT/'vendor',ROOT/'vendor/backbones'):
    sys.path.insert(0,str(item))
CONFIG_PATH=ROOT/'configs/d222.json'
config=json.loads(CONFIG_PATH.read_text(encoding='utf-8-sig'))
# Configure these paths for another machine; no secrets belong in this notebook.
for key in ('prepared_root','tokenizer','phase2_data','output_root','phase1_test','scorer_cache'):
    print(key,config[key])


## 1. Architecture and independent budgets

24 blocks, width 896, 14 query heads and 2 KV heads, SwiGLU 2304, RoPE and pre-RMSNorm, tied vocabulary embedding/output. Eighteen blocks use a true 512-token local window; every fourth block is globally causal. Local attention restricts the actual query/KV computation to tiles/halos. The cache stores two KV heads; cropped local-cache length does not replace the absolute RoPE position counter.

The nominal parameter formula includes embeddings, Q/K/V/O projections, FF layers and norms. It does not include nonexistent positional embeddings. Actual unique parameter count is checked at runtime.


In [ ]:
from legal_d.model import DConfig
cfg=DConfig(**config['model'])
cfg.validate()
print('Nominal unique parameters:',cfg.parameter_formula())
print('Requested supervised pretraining tokens:',config['pretrain']['target_tokens'])
print('Local/global blocks:',cfg.layers-cfg.layers//cfg.global_every,cfg.layers//cfg.global_every)
assert cfg.parameter_formula()==222_079_872


## 2. Data and leakage boundaries

Use the four official legal sources and the existing own-BPE preparation cache. The cache is SHA-verified, has train/dev/confirmation splits and duplicate-aware document provenance. Training supervises all eligible next tokens, including EOS; padding is ignored. Counts of input tokens and supervised target tokens are logged separately, including source contributions.

Wikipedia is disabled: the supplied FAQ permits external corpora, but the live document says official data only. Add any later permitted mixture in a separate run with source/license/token/split provenance. Never load external pretrained participant weights.


In [ ]:
from legal_d.data import verify_prepared
provenance=verify_prepared(config)
print(json.dumps(provenance,indent=2))


## 3. Runtime gates before expensive training

Check causal independence, cropped-cache/absolute-position parity, the real 222M count, finite real forward/backward gradients, attention profiler events, CRF gradients and the empty-document contract. These gates establish runnable code; they do not establish accuracy or a completed pretrained model. The small parity fixture is never submitted.


In [ ]:
from legal_d.preflight import preflight
preflight_record=preflight(config)
print(json.dumps(preflight_record,indent=2))


## 4. Pretrain and resume

AdamW (0.9, 0.95), 0.1 weight decay, warmup/cosine and gradient clipping are initial candidates. The first measured interval calibrates tokens/second and the deadline budget. The requested 300M token budget may be reduced to leave three hours for NER. A 50M pilot is an assessment milestone, not a guaranteed adequate foundation.

`latest.pt` includes optimizer/scaler/RNG/sampler state and exact provenance. `best.pt` is preliminary held-out LM-loss selection. Candidate checkpoints remain for later BERTScore selection. A changed config/data/tokenizer/source fails resume rather than silently mixing experiments. Run the cells manually OR use the detached queue; do not start both.


In [ ]:
from legal_d.pretrain import pretrain
lm_record=pretrain(config,Path(config['output_root'])/'pretrain',resume=True)
print(json.dumps(lm_record,indent=2))


## 5. Phase 2: exact legal NER

NER loads the real selected D checkpoint. The decoder representations are aligned back to every original character, concatenated with character embeddings, processed by a bidirectional character GRU, and scored with a learned constrained BIO CRF. An auxiliary start/end loss targets boundary mistakes. Training-only weights/sampling emphasize rare labels; validation is complete and unweighted.

The 58/12 document split is fixed. Two full-fine-tuning learning-rate/dropout candidates receive a bounded pilot; the better strict macro-F1 candidate continues. LoRMA is a separate adaptation experiment. Test annotations and private leaderboard answers are never used for selection. The official empty test document must map to `[]`.


In [ ]:
from legal_d.ner_train import run_ner
ner_records=run_ner(config,lm_record['checkpoint'])
for record in ner_records:
    print(record['method'],record['validation']['macro_f1'],record['beats_saved_validation_baseline'],record['submission'])


## 6. Phase 1-style comparison after NER delivery

The queue delivers NER ZIPs first. It then scores saved checkpoint/decoding candidates on held-out continuation prompts using official BERTScore F1, ROUGE-L and BLEU, and measures the selected candidate on untouched confirmation. Finally it generates all 1,000 official prompts with exactly 128 complete words each, without filler or truncating prompts.

Pretraining CE selection and measured BERTScore selection are reported separately. The pinned RoBERTa cache is the official evaluation model only, not a training teacher or participant backbone. Missing scorer assets block metrics explicitly. The resulting Phase 1-format archive is marked benchmark-only because the submission phase is closed.


In [ ]:
from legal_d.lm_eval import run_lm_benchmark
benchmark_record=run_lm_benchmark(config)
print(json.dumps(benchmark_record,indent=2))


## 7. Detached GPU1 alternative and monitoring

On stud1, instead of executing training cells, run the source's GPU1 launcher after the runtime preflight. It verifies the previous A0 completion and that GPU1 has no compute process, starts only this queue, and preserves all unrelated jobs. The local downloader independently copies verified NER ZIPs as soon as they exist, even while the closed Phase 1 benchmark continues.

Keep the local computer awake/online for immediate downloads. Remote jobs survive SSH/VS Code disconnects. Source ZIPs and complete checkpoint archives are records, not Codabench submissions.


In [ ]:
# Alternative only; leave False if training cells above were executed.
START_DETACHED_QUEUE=False
if START_DETACHED_QUEUE:
    import subprocess
    subprocess.run(['bash','scripts/run_gpu1.sh'],check=True)
run_root=Path(config['output_root'])
if (run_root/'queue_status.json').exists():
    print((run_root/'queue_status.json').read_text())


## 8. Result contract and honest conclusions

Phase 2 ZIPs: root `prediction.json`, all 35 exact official IDs, integer character start/end offsets and 12 labels, no duplicate keys/spans, CRC/SHA verified. Phase 1 benchmark ZIP: root `prediction.json`, 1,000 exact IDs, every text exactly 128 words.

Report achieved token budgets, actual hardware, held-out per-label scores, failures and whether D improves the saved baseline. Bigger size, sparse attention, a successful backward pass or a valid ZIP is not a leaderboard result. Keep the earlier best submissions when a candidate scores worse. The live brief's Important Instructions require six Phase 2+ experiments while its body still says four; clarify this inconsistency and plan six meaningful methods/architectures.
